In [12]:
import pandas as pd
from ortools.sat.python import cp_model

# importation des jeux de données
df_objet = pd.read_csv('../data/objets.csv', index_col=0)
df_capacite = pd.read_csv('../data/parametres.csv', index_col=0)

In [13]:
# définition des données
objets = list(df_objet.index)
capacite_max = df_capacite.iloc[0,0]
poids = df_objet['poids'].to_dict()
valeur = df_objet['valeur'].to_dict()

In [29]:
# modèle
model = cp_model.CpModel()

# variables de décision
x = {}
for objet in objets:
    x[objet] = model.NewBoolVar(f'x_{objet}')

# contraintes 
model.Add(
    cp_model.LinearExpr.WeightedSum([x[i] for i in objets], [poids[i] for i in objets]) <= capacite_max
)

# fonction objectif
model.maximize(
    cp_model.LinearExpr.WeightedSum([x[i] for i in objets], [valeur[i] for i in objets])
)

# résolution
solver = cp_model.CpSolver()
status = solver.solve(model)

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print(f"Valeur totale: {solver.objective_value}\n")
    for i, var in x.items():
        if solver.value(var) == 1:
            print(f"{i}: sélectionné")
    print(f"Poids total    : {sum(poids[i] for i in objets if solver.value(x[i]) == 1)}")   
    print(f"Statut         : {solver.StatusName()}")
else:
    print("No solution found.")

Valeur totale: 85.0

Tente: sélectionné
Sac_de_couchage: sélectionné
Rechaud: sélectionné
Nourriture: sélectionné
Trousse_medicale: sélectionné
Vetements: sélectionné
Lampe_frontale: sélectionné
Carte_boussole: sélectionné
Corde: sélectionné
Gourde: sélectionné
Appareil_photo: sélectionné
Poids total    : 50
Statut         : OPTIMAL
